<a href="https://colab.research.google.com/github/beingAnujChaudhary/Customer-Churn-Prediction-Retention-Prioritization/blob/main/notebooks/05_prioritization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %% [markdown]
# # 🎯 Stage 8: Prioritization Engine
# *Notebook: 05_prioritization.ipynb*
#
# **Purpose**: Combine churn probability with customer value to prioritize retention efforts.
# **Output**: `output/prioritized_customers.csv` for Power BI.

# %% [code]
import pandas as pd
import numpy as np
import os

# Setup output directory
os.makedirs("output", exist_ok=True)
print("✓ Environment ready")

✓ Environment ready


In [2]:
# %% [code]
# LOAD DATA
# 1. Load Predictions (from Stage 7)
df_preds = pd.read_csv("output/predictions_with_shap.csv")

# 2. Load Processed Data (from Stage 6) - contains all features
df_full = pd.read_csv("data/processed/customers_v3.csv")


# 3. Extract Test Set
# We filter for the test split to match the predictions we just made
df_test = df_full[df_full["split"] == "test"].drop(columns=["split"]).reset_index(drop=True)

# 4. Merge Predictions with Customer Data
# We assume row order is preserved from the test set generation
df_eng = df_test.copy()
df_eng["churn_probability"] = df_preds["churn_probability"]
df_eng["risk_tier_raw"] = df_preds["predicted_churn"] # 0 or 1

print(f"✅ Data loaded: {len(df_eng)} customers in Test Set")

✅ Data loaded: 2026 customers in Test Set


In [3]:
# %% [code]
# DEFINE CUSTOMER VALUE
# Proxy: Total Transaction Amount (Total_Trans_Amt)
df_eng["customer_value"] = df_eng["Total_Trans_Amt"]

# Calculate Value Threshold (Top 30% are "High Value")
value_threshold = df_eng["customer_value"].quantile(0.70)

def assign_value_tier(value):
    return "High Value" if value >= value_threshold else "Standard Value"

df_eng["value_tier"] = df_eng["customer_value"].apply(assign_value_tier)

# DEFINE RISK TIERS
# Based on Churn Probability
def assign_risk_tier(prob):
    if prob > 0.70:
        return "High Risk"
    elif prob > 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"

df_eng["risk_tier"] = df_eng["churn_probability"].apply(assign_risk_tier)

print(f"✅ Value Threshold (Top 30%): ${value_threshold:,.2f}")
print("✅ Risk & Value Tiers assigned")

✅ Value Threshold (Top 30%): $4,579.50
✅ Risk & Value Tiers assigned


In [4]:
# %% [code]
# CALCULATE PRIORITY SCORE
# Formula: Churn Probability × Customer Value
# This ensures we prioritize HIGH RISK + HIGH VALUE customers first.
df_eng["priority_score"] = df_eng["churn_probability"] * df_eng["customer_value"]

# SORT BY PRIORITY (Highest first)
df_eng = df_eng.sort_values(by="priority_score", ascending=False)

print("✅ Priority scores calculated and sorted")

✅ Priority scores calculated and sorted


In [5]:
# %% [code]
# GENERATE TOP 100 AT-RISK CUSTOMERS
# Focus on High/Medium risk customers with high value
target_customers = df_eng[df_eng["risk_tier"].isin(["High Risk", "Medium Risk"])].head(100)

# Select columns relevant for the Sales Team
sales_cols = [
    "customer_value", "churn_probability", "risk_tier", "value_tier",
    "priority_score", "Total_Trans_Ct", "Months_Inactive_12_mon",
    "Total_Relationship_Count", "Contacts_Count_12_mon"
]

sales_list = target_customers[sales_cols].copy()

# Save to CSV
sales_list.to_csv("output/top_100_at_risk.csv", index=False)
print("✅ Saved: output/top_100_at_risk.csv")
print(f"   Count: {len(sales_list)}")

✅ Saved: output/top_100_at_risk.csv
   Count: 100


In [6]:
# %% [code]
# EXPORT FULL DATASET FOR DASHBOARD
# Power BI needs the whole population to calculate aggregates and show distributions
full_export_cols = [
    "customer_value", "churn_probability", "risk_tier", "value_tier",
    "priority_score", "Total_Trans_Ct", "Months_Inactive_12_mon"
]

df_final_export = df_eng[full_export_cols].copy()
df_final_export.to_csv("output/prioritized_customers.csv", index=False)
print("✅ Saved: output/prioritized_customers.csv (For Power BI)")

✅ Saved: output/prioritized_customers.csv (For Power BI)
